# Part 3 · Notebook 01 — Money, ticks and time zones

**Sessions:** S1 (Types, numbers & money) · S3 (Collections, dates & time zones) · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. See why floats are wrong for money and prices, and use `Decimal` instead.
2. Round prices to an instrument's tick and compute a commission exactly.
3. Convert the New York open to UTC and watch daylight saving move it.
4. Keep a rolling window with `deque`, and feel why dict lookups beat list scans.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

## 1. Floats are binary approximations

In [ ]:
print(0.1 + 0.2, 0.1 + 0.2 == 0.3)
print(Decimal("0.1") + Decimal("0.2"), Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))
print(Decimal(0.1))                       # built from a float: the float's error comes along
total_f, total_d = 0.0, Decimal("0")
for _ in range(1_000_000):                # a million one-cent fees
    total_f += 0.01
    total_d += Decimal("0.01")
print(f"float total:   {total_f!r}")
print(f"Decimal total: {total_d!r}")

## 2. Rounding to the tick

Exchanges only accept prices on the instrument's tick grid. Round **in Decimal**, half up.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from decimal import ROUND_HALF_UP

def round_to_tick(price: Decimal, tick: Decimal) -> Decimal:
    # ✍️ divide by the tick, quantize to a whole number with ROUND_HALF_UP, multiply back
    return ...

tests = [(Decimal("101.237"), Decimal("0.05")), (Decimal("4512.30"), Decimal("0.25")), (Decimal("0.123456"), Decimal("0.0001"))]
mine = [round_to_tick(px, tk) for px, tk in tests]
mine = p.check("round_to_tick", mine, [p.round_to_tick(px, tk) for px, tk in tests])
mine

In [ ]:
# The float version looks fine... until it doesn't
def round_to_tick_float(price, tick):
    return round(price / tick) * tick

bad = [(x, round_to_tick_float(x, 0.05)) for x in (101.237, 1.005, 0.35)]
bad                                          # values like 0.35000000000000003 are OFF the tick grid

## 3. Commission in Decimal

IB fixed pricing (lesson-plan figures): **$0.005 per share, minimum $1.00, maximum 1% of trade value**, rounded to the cent.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def commission(shares: int, price: Decimal) -> Decimal:
    raw = Decimal("0.005") * shares
    cap = Decimal("0.01") * shares * price
    # ✍️ apply the $1.00 minimum and the 1% cap, then quantize to Decimal("0.01") with ROUND_HALF_UP
    return ...

trades = [(100, Decimal("50")), (1000, Decimal("0.5")), (500, Decimal("20")), (10_000, Decimal("3.1"))]
fees = [commission(n, px) for n, px in trades]
fees = p.check("IB fixed commission", fees, [p.ib_fixed_commission(n, px) for n, px in trades])
fees

In [ ]:
rng = np.random.default_rng(0)
sizes, prices_ = rng.integers(1, 5000, 10_000), rng.uniform(1, 500, 10_000).round(2)
dec = [p.ib_fixed_commission(int(n), Decimal(str(px))) for n, px in zip(sizes, prices_)]
flt = [round(min(max(0.005 * n, 1.0), 0.01 * n * px), 2) for n, px in zip(sizes, prices_)]
differ = sum(Decimal(str(f)) != d for f, d in zip(flt, dec))
print(f"Decimal total: ${sum(dec):,}   float total: ${sum(flt):,.10f}")
print(f"The float version disagrees with exact half-up rounding on {differ:,} of 10,000 trades: round() on a float "
      "rounds a binary approximation, half to even (1.005 → 1.0). Pennies, multiplied by every trade.")

## 4. Time zones: store in UTC, think in exchange time

The US open is 09:30 in New York, which is **14:30 UTC in winter and 13:30 UTC in summer**.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from datetime import date, datetime, time
from zoneinfo import ZoneInfo
NY, UTC = ZoneInfo("America/New_York"), ZoneInfo("UTC")

def market_open_utc(d: date) -> datetime:
    # ✍️ combine d with 09:30, attach tzinfo=NY, convert to UTC
    return ...

days = [date(2025, 1, 15), date(2025, 7, 15), date(2025, 3, 10)]
opens = [market_open_utc(d) for d in days]
opens = p.check("market open in UTC", opens, [p.market_open_utc(d) for d in days])
opens

In [ ]:
year = pd.bdate_range("2025-01-01", "2025-12-31")
hours = [p.market_open_utc(d.date()).hour + p.market_open_utc(d.date()).minute / 60 for d in year]
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.step(year, hours, where="post")
ax.set(title="The US open in UTC through 2025", ylabel="UTC hour", yticks=[13.5, 14.5], yticklabels=["13:30", "14:30"])
plt.show()
naive = datetime(2025, 3, 10, 9, 30)
print("A naive datetime has no zone:", naive.tzinfo, "- comparing it with an aware one raises TypeError.")

## 5. Collections: a rolling window and O(1) lookups

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from collections import deque

def rolling_mean(values, n):
    win, out = deque(maxlen=n), []
    for v in values:
        win.append(v)
        # ✍️ append the mean of the window once it holds n values, else None
        ...
    return out

vals = [10, 11, 12, 13, 14, 15]
rm = rolling_mean(vals, 3)
rm = p.check("rolling mean with deque", rm, p.rolling_mean_deque(vals, 3))
rm

In [ ]:
sizes_ = [1_000, 10_000, 100_000]
rows = []
for n in sizes_:
    symbols = [f"S{i}" for i in range(n)]
    as_list, as_set = symbols, set(symbols)
    rows.append({"n": n, "list scan (µs)": p.timeit(lambda: "missing" in as_list, number=20) * 1e6,
                 "set lookup (µs)": p.timeit(lambda: "missing" in as_set, number=20) * 1e6})
pd.DataFrame(rows).set_index("n").round(2)

## Questions
1. Where in a trading system is a float fine, and where must you use `Decimal`?
2. Why is `Decimal(0.1)` different from `Decimal("0.1")`?
3. A strategy stores bar times as naive local times. What goes wrong twice a year?

**Graded version:** `labs/part03/week07_basics` (`round_to_tick`, the commission, New York/UTC times).